# ArchiMind dataset processing (Colab driver)

Setup guide: `docs/developer/DATASET_COLAB.md`. This notebook only calls the tested
`python -m archimind.datasets` commands. Edit **only** the configuration cell.
Re-running from the top with the same `RUN_ID` resumes: finished stages are skipped.

In [33]:
# --- Configuration (edit here only) ---
RUN_ID = "pilot-001"  # reuse the same id to resume after a disconnect
ARCHIMIND_COMMIT = "77ea0264dfdac67a2f9a930632bb8216e0bbb8b2"  # exact commit, never a branch
REPO = "ngstephen1/ArchiMind"
WORK = "/content/work"  # fast local disk (lost when the runtime ends)
DRIVE_ROOT = "/content/drive/MyDrive/ArchiMind-data"  # persistent backup
FLOOR_GIB = 4  # stop below this much free local disk

In [34]:
# Mount Drive and create the run layout.
import datetime
import os
import pathlib
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

RUN_DRIVE = pathlib.Path(DRIVE_ROOT) / "runs" / RUN_ID
for d in ("raw", "stages", "packs", "receipts"):
    (pathlib.Path(WORK) / d).mkdir(parents=True, exist_ok=True)
for d in ("stages", "packs", "receipts"):
    (RUN_DRIVE / d).mkdir(parents=True, exist_ok=True)
print("local work:", WORK)
print("drive backup:", RUN_DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
local work: /content/work
drive backup: /content/drive/MyDrive/ArchiMind-data/runs/pilot-001


In [35]:
# Load secrets into this process only. Values are never printed.
from google.colab import userdata

for name in ("GH_TOKEN", "HF_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: loaded")
    except Exception:
        print(
            f"{name}: not available (add it under the key icon and enable notebook access)"
        )

GH_TOKEN: loaded
HF_TOKEN: loaded
KAGGLE_USERNAME: loaded
KAGGLE_KEY: loaded


In [36]:
# Install the pinned ArchiMind commit. Git output is hidden so the token never appears.
import re

assert re.fullmatch(r"[0-9a-f]{40}", ARCHIMIND_COMMIT), (
    "a48f42d75c2b387b2fe8a971c52274e86c05e407"
)
SRC = "/content/ArchiMind"
PLAIN = f"https://github.com/{REPO}.git"
AUTHED = (
    f"https://x-access-token:{os.environ.get('GH_TOKEN', '')}@github.com/{REPO}.git"
)


def _git(*args, cwd=SRC):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(
            f"git {args[0]} failed; output hidden to protect the token. Check GH_TOKEN."
        )


if not os.path.isdir(f"{SRC}/.git"):
    _git("clone", "--quiet", "--no-checkout", AUTHED, SRC, cwd="/content")
try:
    _git("remote", "set-url", "origin", AUTHED)
    _git("fetch", "--quiet", "origin", ARCHIMIND_COMMIT)
    _git("checkout", "--quiet", "--detach", ARCHIMIND_COMMIT)
finally:
    _git("remote", "set-url", "origin", PLAIN)  # never leave the token in .git/config

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", SRC],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "pydantic==2.14.1"], check=True
)
os.environ["ARCHIMIND_COMMIT"] = ARCHIMIND_COMMIT
print("installed", ARCHIMIND_COMMIT)

RuntimeError: git fetch failed; output hidden to protect the token. Check GH_TOKEN.

In [20]:
# Helpers: run a dataset command, checkpoint to Drive, restore from Drive.
def ds(*args):
    r = subprocess.run(
        [sys.executable, "-m", "archimind.datasets", *map(str, args)],
        capture_output=True,
        text=True,
    )
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"archimind.datasets {args[0]} exited {r.returncode}")


def checkpoint():
    for d in ("stages", "packs", "receipts"):
        ds("sync", f"{WORK}/{d}", RUN_DRIVE / d)


def restore():
    for d in ("stages", "packs", "receipts"):
        ds("sync", RUN_DRIVE / d, f"{WORK}/{d}")

In [21]:
# Restore earlier progress for this RUN_ID (no-op on a fresh run).
restore()

{
  "bytes_copied": 0,
  "copied": 0,
  "unchanged": 0
}

{
  "bytes_copied": 0,
  "copied": 0,
  "unchanged": 0
}

{
  "bytes_copied": 0,
  "copied": 0,
  "unchanged": 0
}



In [22]:
# Environment receipt and free-space check (fails below FLOOR_GIB).
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ds(
    "env",
    "--data-root",
    WORK,
    "--floor-gib",
    FLOOR_GIB,
    "--out",
    f"{WORK}/receipts/env-{stamp}.json",
)
checkpoint()

{
  "above_floor": true,
  "archimind_commit": "a48f42d75c2b387b2fe8a971c52274e86c05e407",
  "archimind_version": "0.1.0",
  "colab": true,
  "floor_bytes": 4294967296,
  "free_bytes": 115491483648,
  "machine": "x86_64",
  "observed_utc": "2026-11-11T21:13:44Z",
  "packages": {
    "numpy": "2.4.6",
    "pydantic": "2.14.1"
  },
  "platform": "linux",
  "python": "3.13.18",
  "schema_version": "1.0.0",
  "secrets_present": {
    "GH_TOKEN": true,
    "HF_TOKEN": true,
    "KAGGLE_KEY": true,
    "KAGGLE_USERNAME": true
  }
}

{
  "bytes_copied": 0,
  "copied": 0,
  "unchanged": 0
}

{
  "bytes_copied": 0,
  "copied": 0,
  "unchanged": 0
}

{
  "bytes_copied": 532,
  "copied": 1,
  "unchanged": 0
}



In [23]:
# Self-check: build and verify a synthetic pack end to end, then checkpoint it.
ds("fixture-pack", f"{WORK}/packs/synthetic-selfcheck")
ds("verify-pack", f"{WORK}/packs/synthetic-selfcheck")
checkpoint()

{
  "snapshot_id": "26743f7429d73f283a4224dd936aa4ec56f9c815f17735bd21fa40008e1357c3",
  "status": "built",
  "synthetic": true
}

{
  "assets": 1,
  "audience": "local",
  "exclusions": 1,
  "pack_id": "synthetic-selfcheck",
  "scenes": 2,
  "snapshot_id": "26743f7429d73f283a4224dd936aa4ec56f9c815f17735bd21fa40008e1357c3",
  "sources": [
    "future_3d",
    "ssr_3dfront"
  ],
  "status": "verified"
}

{
  "bytes_copied": 0,
  "copied": 0,
  "unchanged": 0
}

{
  "bytes_copied": 5507,
  "copied": 6,
  "unchanged": 0
}

{
  "bytes_copied": 0,
  "copied": 0,
  "unchanged": 1
}



## P1: probe (inventory and yield)

Outputs go to `WORK/stages/p1` and are checkpointed to Drive. SSR and MIDI pins
come from the P1 probe of 2026-11-06. Re-running skips files that already match
their pinned SHA-256. Only the FUTURE cell needs Kaggle credentials.

In [24]:
# SSR-3DFRONT: pinned revision; ~7.4 MB of parquet in total.
P1 = f"{WORK}/stages/p1"
SSR_REPO = "gradient-spaces/SSR-3DFRONT"
SSR_REV = "b606183e56d972f85850125f9191932cbea654c3"
SSR_SHARDS = {  # path -> SHA-256 published by Hugging Face (LFS oid)
    "data/train-00000-of-00001.parquet": "aedbb1f9448614299ecc6c40e6ccf4f9b7441feee42275d7fcfb06690de038e1",
    "data/val-00000-of-00001.parquet": "38e99958698c2633199e041f68cba51b91f2db1255718ea894a6db8ee323fe40",
    "data/test-00000-of-00001.parquet": "8970830a27fb3dafb5e89d324f5f3666b3defd489e79345d314d455d00a1331b",
}
HF = "https://huggingface.co/datasets"
ds("probe-hf", SSR_REPO, "--revision", SSR_REV, "--out", f"{P1}/ssr_inventory.json")
local = []
for path, sha in SSR_SHARDS.items():
    dest = f"{WORK}/raw/ssr/{path.rsplit('/', 1)[-1]}"
    ds(
        "fetch",
        f"{HF}/{SSR_REPO}/resolve/{SSR_REV}/{path}",
        dest,
        "--max-bytes",
        8_000_000,
        "--sha256",
        sha,
    )
    local.append(dest)
ds("probe-ssr", *local, "--out", f"{P1}/ssr_probe.json")
checkpoint()

{
  "current_revision": "b606183e56d972f85850125f9191932cbea654c3",
  "file_count": 6,
  "files": [
    {
      "path": ".gitattributes",
      "sha256": null,
      "size_bytes": 2461
    },
    {
      "path": "README.md",
      "sha256": null,
      "size_bytes": 4930
    },
    {
      "path": "data/test-00000-of-00001.parquet",
      "sha256": "8970830a27fb3dafb5e89d324f5f3666b3defd489e79345d314d455d00a1331b",
      "size_bytes": 858995
    },
    {
      "path": "data/train-00000-of-00001.parquet",
      "sha256": "aedbb1f9448614299ecc6c40e6ccf4f9b7441feee42275d7fcfb06690de038e1",
      "size_bytes": 5698138
    },
    {
      "path": "data/val-00000-of-00001.parquet",
      "sha256": "38e99958698c2633199e041f68cba51b91f2db1255718ea894a6db8ee323fe40",
      "size_bytes": 848612
    },
    {
      "path": "raw_scenes.zip",
      "sha256": "8a38f6309d447a4abd03455762aaf8c8cc1c0f4abc973ffa1f0f740d55fcf238",
      "size_bytes": 21881708
    }
  ],
  "gated": false,
  "last_modified":

In [25]:
# MIDI-3D: ID lists (a few MB) plus a streamed 1 GiB prefix listing of the test-scene
# archive. The prefix is read and discarded; nothing from it is stored.
MIDI_REPO = "huanngzh/3D-Front"
MIDI_REV = "156fc5312ea613d4e92cbf5da1fddfa61bbad3b2"
MIDI = f"{HF}/{MIDI_REPO}/resolve/{MIDI_REV}"
ds("probe-hf", MIDI_REPO, "--revision", MIDI_REV, "--out", f"{P1}/midi_inventory.json")
for name in (
    "midi_room_ids.json",
    "midi_furniture_ids.json",
    "midi_test_room_ids.json",
    "midi_test_furniture_ids.json",
):
    ds("fetch", f"{MIDI}/{name}", f"{P1}/midi/{name}", "--max-bytes", 5_000_000)
ds(
    "probe-tar-prefix",
    f"{MIDI}/3D-FRONT-TEST-SCENE.tar.gz",
    "--max-bytes",
    1024**3,
    "--out",
    f"{P1}/midi_test_scene_prefix.json",
)
checkpoint()

 {
      "path": "3D-FRONT-SURFACE.partam",
      "sha256": "df2f134c2f15fb8db316a414e832f9d837e699e5b6b6944e70eb51e108eb613c",
      "size_bytes": 42949672960
    },
    {
      "path": "3D-FRONT-SURFACE.partan",
      "sha256": "2b74dda5a127e6fba18b17b12f00ba07de18496b929a46e0970dd6918d032bb4",
      "size_bytes": 42949672960
    },
    {
      "path": "3D-FRONT-SURFACE.partao",
      "sha256": "31bf36bf0d7597f8848591f9468357f003f9ccc8b5c7ba171f62dceed60eb129",
      "size_bytes": 42949672960
    },
    {
      "path": "3D-FRONT-SURFACE.partap",
      "sha256": "de7c07524e7953e43c276c54ddefc7529ca82e95777ee25cde1ecca07ee82992",
      "size_bytes": 42949672960
    },
    {
      "path": "3D-FRONT-SURFACE.partaq",
      "sha256": "332d6a589b54469020ae3e028f41a7cad415dda777fb1502054a226022baba85",
      "size_bytes": 42949672960
    },
    {
      "path": "3D-FRONT-SURFACE.partar",
      "sha256": "3777a338071f91ed3353c9e6a8e81d36226fe79a4725514c99d868167b11deaa",
      "size_bytes": 42

In [27]:
import os, subprocess
ref = "tobetheonly/3d-future-model"
u, k = os.environ.get("KAGGLE_USERNAME",""), os.environ.get("KAGGLE_KEY","")
print("user set:", bool(u), "| key set:", bool(k), "| key length:", len(k))
def code(auth):
    cmd = ["curl","-s","-o","/dev/null","-w","%{http_code}"]
    if auth: cmd += ["-u", f"{u}:{k}"]
    cmd += [f"https://www.kaggle.com/api/v1/datasets/view/{ref}"]
    return subprocess.run(cmd,capture_output=True,text=True).stdout
print("view, no auth:", code(False))
print("view, with auth:", code(True))
print("list files, with auth:", subprocess.run(["curl","-s","-o","/dev/null","-w","%{http_code}","-u",f"{u}:{k}",
    f"https://www.kaggle.com/api/v1/datasets/list/files/{ref}?pageSize=1"],capture_output=True,text=True).stdout)

user set: True | key set: True | key length: 37
view, no auth: 404
view, with auth: 404
list files, with auth: 404


In [28]:
# 3D-FUTURE (Kaggle mirror): version, full file list and model_info.json, the
# authoritative jid -> category table. Needs KAGGLE_USERNAME and KAGGLE_KEY.
import json

FUTURE_REF = "tobetheonly/3d-future-model"
ds("probe-kaggle", FUTURE_REF, "--out", f"{P1}/future_inventory.json")
inventory = json.load(open(f"{P1}/future_inventory.json"))
print("model_info candidates:", inventory.get("model_info"))
for i, entry in enumerate(inventory.get("model_info") or []):
    ds(
        "fetch",
        entry["url"],
        f"{P1}/future/model_info_{i}.download",
        "--max-bytes",
        64 * 1024**2,
        "--kaggle-auth",
    )
checkpoint()


Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/ArchiMind/src/archimind/datasets/__main__.py", line 5, in <module>
    sys.exit(main())
             ~~~~^^
  File "/content/ArchiMind/src/archimind/datasets/cli.py", line 494, in main
    return args.func(args)
           ~~~~~~~~~^^^^^^
  File "/content/ArchiMind/src/archimind/datasets/cli.py", line 174, in _cmd_probe_kaggle
    payload = kaggle_dataset_view(args.ref)
  File "/content/ArchiMind/src/archimind/datasets/remote.py", line 114, in kaggle_dataset_view
    info = _get_json(f"{KAGGLE_API}/datasets/view/{ref}", opener=opener)
  File "/content/ArchiMind/src/archimind/datasets/remote.py", line 39, in _get_json
    with opener(request, timeout) as response:
         ~~~~~~^^^^^^^^^^^^^^^^^^
  File "/content/ArchiMind/src/archimind/datasets/remote.py", line 30, in _default_opener
    return urllib.request.urlopen(request, tim

RuntimeError: archimind.datasets probe-kaggle exited 1

**Hand-back:** download `MyDrive/ArchiMind-data/runs/<RUN_ID>/stages/p1/` and place
it at `E:rchimind-data\p1\` (outside the Git checkout). P2 reads it from there.

In [ ]:
# Finalize before closing the tab: checkpoint, then make Drive finish writing.
checkpoint()
drive.flush_and_unmount()
print("Drive flushed. Safe to close.")